### Part one : load data and split it

In [21]:
import pandas as pd 
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.decomposition import PCA

In [22]:
path = "data/penguins_with_splits.csv"
data = pd.read_csv(path)
row , col = data.shape
print(f"Rows: {row}, Columns: {col}")
data.head()

Rows: 344, Columns: 9


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,split
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,train
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,val
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,train
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007,train
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,train


In [23]:
test = data[data['split'] == 'test']
val = data[data['split'] == 'val']
train = data[data['split'] == 'train']

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (241, 9)
Validation: (69, 9)
Test: (34, 9)


### Part tow : Build a Preprocessing Pipeline

In [24]:
numeric_features = [
  "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "year"
]

categorical_features = ["island", "sex"]

def add_bill_ratio(x):
  ratio = x[:, 0] / x[:, 1]
  return np.column_stack((x, ratio))

## build pipeline

In [ ]:
numeric_pipeline = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("ratio", FunctionTransformer(add_bill_ratio, validate=False)),
  ("scaler", StandardScaler()),
  ("pca", PCA(n_components=0.95))
]
)

categorical_pipeline = Pipeline([
  ("imputer",SimpleImputer(strategy="most_frequent")),
  ("encoder", OneHotEncoder(
    handle_unknown="ignore" ,
    sparse_output=False
  ))
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, ["island"])
])

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

### Part 3 — Apply the Pipeline

In [26]:
input_features = numeric_features + ["island"]

x_train = train[input_features]
x_val = val[input_features]
x_test = test[input_features]

In [27]:
train_processed = pipeline.fit_transform(x_train)
test_processed = pipeline.transform(x_test)
val_processed = pipeline.transform(x_val)

In [28]:
print("Train:", train_processed.shape)
print("Validation:", val_processed.shape)
print("Test:", test_processed.shape)

Train: (241, 7)
Validation: (69, 7)
Test: (34, 7)


In [30]:
print("Missing values:", np.isnan(train_processed).sum())

pca = pipeline.named_steps["preprocessor"] \
    .named_transformers_["num"] \
    .named_steps["pca"]

print("PCA Components:", pca.n_components_)
print("Explained Variance:", pca.explained_variance_ratio_.sum())

Missing values: 0
PCA Components: 4
Explained Variance: 0.9813402669404832


### Part 4: Save the processed dataset